In [5]:
import kagglehub
import pandas as pd
import numpy as np
# Download latest version
# path = kagglehub.dataset_download("rohanrao/formula-1-world-championship-1950-2020")

# print("Path to dataset files:", path)

In [7]:
from typing import TypedDict
from langgraph.graph import StateGraph, END
from dotenv import load_dotenv
load_dotenv()

import getpass
import os

if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API key: ")

In [6]:
import sqlite3

conn = sqlite3.connect("f1_data.db")

csv_files = ["circuits.csv","constructor_results.csv","constructor_standings.csv",
             "drivers.csv", "races.csv", "results.csv", "constructors.csv",
             "driver_standings.csv",'drivers.csv','lap_times.csv',
             "pit_stops.csv","qualifying.csv","races.csv",'results.csv','seasons.csv',
             "sprint_results.csv",'status.csv']

for file in csv_files:
    df = pd.read_csv(file)
    # Clean column names (remove spaces/special chars for SQL safety)
    df.columns = [c.replace(' ', '_').lower() for c in df.columns]
    table_name = file.replace(".csv", "")
    df.to_sql(table_name, conn, if_exists="replace", index=False)

# Version 1: Multi-table SQLite LangGraph pipeline

Architecture:

User input -> Generate SQL query -> Fetch data from SQLite -> Summarize result -> Return answer

In [95]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0,
    max_tokens=None,
    reasoning_format="parsed",
    timeout=None,
    max_retries=2
)

In [96]:
class State(TypedDict):
    user_query:str
    sql_code:str
    sql_code_answer:str
    summary:str

Define the graph state and node functions for the V1 pipeline.

In [97]:
# V1 uses plain LangGraph node functions, not LangChain tool wrappers.

The nodes below generate SQL, execute it against the local SQLite database, and summarize the result.

In [120]:
conn = sqlite3.connect('f1_data.db')
cursor = conn.cursor()


table_names = [c.replace(".csv","") for c in csv_files]
schema_info_all = []
for table in table_names:
    cursor.execute(f"PRAGMA table_info({table})")

    columns = cursor.fetchall()
    schema_info_all.append([[col[1],col[2]] for col in columns])

table_schema_info={}
for i in range(len(table_names)):
        table_schema_info[table_names[i]] = schema_info_all[i]


def query_generator(state:State):
    """Function for transforming user input into SQL query"""
    prompt = f"""Generate SQL code for this query - {state['user_query']}, using the provided context in a dictionary: 
    where the keys are table_names and the values are the schema in a list form.
    {table_schema_info}. Respond only with the SQL query."""
    output = llm.invoke(prompt)
    return {'sql_code':output.content}


def run_sqlite_query(state:State):
    """
    Executes a SQL query against the F1 dataset and returns the results.
    Use this to join tables or aggregate F1 statistics.
    """
    conn = sqlite3.connect("f1_data.db")
    try:
        cursor = conn.cursor()
        cursor.execute(state['sql_code'])
        result = cursor.fetchall()
        return {'sql_code_answer':result}
    except Exception as e:
        return {'sql_code_answer': f"Error: {str(e)}"}
    finally:
        conn.close()
        


def summarizer(state:State):
    """ Provides a proper summary for the SQL query and its answer for the user."""
    prompt = f"""SYSTEM INSTRUCTION:
            You are an F1 Data Analyst. Your task is to take a user's question and the 
            corresponding SQL data results to provide a polished, expert response.

            INPUT DATA:
            1. User Question: {state['user_query']}
            2. Executed SQL: {state['sql_code']}
            3. SQL Result: {state['sql_code_answer']}

            GUIDELINES:
            - Directness: Answer the user's question in the first sentence.
            - Accuracy: Use ONLY the data provided in the 'SQL Result'. Do not hallucinate external stats.
            - Professionalism: If the result is a list of names or numbers, format them clearly (e.g., use bullet points for multiple drivers).
            - Context: If the SQL result is empty, politely explain that no records were found for that specific query in the F1 database.
            - Detail: Mention the specific driver names and values exactly as they appear in the result.

            RESPONSE FORMAT:
            "Based on the race data, [Your natural language answer here]."
            """
    output = llm.invoke(prompt)
    return {'summary':output.content}


In [121]:
builder = StateGraph(State)
builder.add_node("query generator",query_generator)
builder.add_node("sqlite3 generator",run_sqlite_query)
builder.add_node("summarization",summarizer)
builder.set_entry_point("query generator")
builder.add_edge("query generator", "sqlite3 generator")
builder.add_edge("sqlite3 generator","summarization")
builder.add_edge("summarization",END)
graph = builder.compile()

result = graph.invoke({"user_query":"tell me who won the 2023 italian GP"})

In [122]:
result

{'user_query': 'tell me who won the 2023 italian GP',
 'sql_code': "SELECT d.forename, d.surname\nFROM races r\nJOIN circuits c ON r.circuitid = c.circuitid\nJOIN results res ON r.raceid = res.raceid\nJOIN drivers d ON res.driverid = d.driverid\nWHERE r.year = 2023\n  AND c.country = 'Italy'\n  AND res.positionorder = 1;",
 'sql_code_answer': [('Max', 'Verstappen')],
 'summary': 'Based on the race data, Max Verstappen won the 2023 Italian GP.'}